# Test `Fluid`

This notebook demonstrates how to use the `Fluid` class provided by
**GTh (Thermofluids)**.

The `Fluid` class provides a convenient interface for obtaining
thermophysical properties of a fluid. The underlying thermophysical
property calculations are performed by the **CoolProp** library.

The notebook illustrates:

- how to create a `Fluid` object;
- how to specify the thermodynamic state;
- how to obtain commonly used thermophysical properties;
- how phase information can be specified when required.

The notebook is intended both as a basic usage example and as a simple
test of the `Fluid` class and its connection to CoolProp.

##Do not modify the bootstrap cells.

## START BOOTSTRAP ##

In [1]:
#@title GTh internal bootstrap. You must execute this cell {display-mode: "form"}
# ============================================================
# GTh - Internal bootstrap
# DO NOT MODIFY
# ============================================================

from pathlib import Path
import sys


def _setup_thermofluids(environment=None):

    if environment == "local":
        here = Path(__file__).resolve().parent
    else:
        here = Path.cwd().resolve()

    # Search for the Thermofluids root
    for path in [here, *here.parents]:

        # Case 1: path is the Thermofluids root
        if (path / "lib").is_dir():
            gth_root = path
            break

        # Case 2: path contains a Thermofluids directory
        if (path / "Thermofluids" / "lib").is_dir():
            gth_root = path / "Thermofluids"
            break

    else:
        raise RuntimeError(
            "Could not find Thermofluids/lib"
        )

    lib_path = gth_root / "lib" / "python"

    if str(lib_path) not in sys.path:
        sys.path.insert(0, str(lib_path))

    return gth_root
print(f"_setup_thermofluids(): Load")

_setup_thermofluids(): Load


In [2]:
# ============================================================
# GTh - Bootstrap
# 1. Detect execution environment
# ============================================================

import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ENVIRONMENT = "colab" if IN_COLAB else "local"

print(f"Environment: {ENVIRONMENT}")

Environment: local


In [3]:
# ============================================================
# GTh - Bootstrap
# 2. Select GTh source
if  IN_COLAB:
   # ============================================================
# GTh - Dependencies
# ============================================================

  import importlib.util
  import subprocess

  if importlib.util.find_spec("CoolProp") is None:

     print("CoolProp not found. Installing...")

     subprocess.check_call([
          sys.executable,
          "-m",
          "pip",
          "install",
         "CoolProp"
     ])
     print("CoolProp installed.")
  else:

      print("CoolProp already installed.")

In [5]:
# ============================================================
# GTh - Bootstrap
# 3. Locate GTh
# ============================================================

if ENVIRONMENT == "local":

    GTH_ROOT =_setup_thermofluids()
    
    print("Thermofluids root detected.")
    print(f"Thermofluids available at: {GTH_ROOT}")

else:

    GTH_ROOT = Path("/content/Thermofluids")

    if not GTH_ROOT.exists():

        print("Cloning Thermofluids from GitHub...")

        subprocess.check_call([
            "git",
            "clone",
            "https://github.com/rvieytes/Thermofluids.git",
            str(GTH_ROOT)
        ])

    print(f"Thermofluids available at: {GTH_ROOT}")

    _setup_thermofluids()


Thermofluids root detected.
Thermofluids available at: /home/roberto/Documentos/itba/libro/github_prod/Thermofluids


## Thermodynamic State


The thermodynamic state of the fluid is defined by its temperature $T$
and pressure $P$. The phase can either be left unspecified or used to
select a saturated state.

- `phase = None`: the phase is determined from $(T,P)$.
- `phase = "liq"`: saturated liquid at the specified temperature. The
  pressure must be consistent with the saturation pressure
  $P_\mathrm{sat}(T)$.
- `phase = "vap"`: saturated vapor at the specified temperature. The
  pressure must be consistent with the saturation pressure
  $P_\mathrm{sat}(T)$.

Therefore, `"liq"` and `"vap"` **do not mean that the fluid phase can be
arbitrarily forced for any given $(T,P)$**. They specify that the state
lies on the saturation curve.

If an inconsistent combination of temperature, pressure and phase is
specified, `Fluid` raises an error rather than silently modifying or
ignoring the input.

For example, specifying

```python
temperature = 300.16
pressure = 10325
phase = "liq"
```

will raise an error because the specified pressure is not the saturation
pressure of water at $300.16\ \mathrm{K}$.

For water at $T=300.16\ \mathrm{K}$ and atmospheric pressure,
$P=101325\ \mathrm{Pa}$, the thermodynamic state is liquid, and we can
use:
```
phase=None
```
**Temperature scale

Temperatures used by CoolProp are absolute temperatures in kelvin.
For development and high-precision calculations, use the exact conversion

T=t+273.15 K;

do not use the rounded value 273 K.

In [6]:
# ============================================================
# Thermodynamic state
# ============================================================
from thermophysics import Fluid
fluid_name = "Water"

temperature = 300.16       # K
pressure = 101325          # Pa

phase = None             # None, "liq" or "vap"
f = Fluid(
    fluid_name,
    temperature=temperature,
    pressure=pressure,
    phase=phase
)
print(f"Fluid object: instantiated ")

Fluid object: instantiated 


In [7]:

print(f"rho        = {f.rho:.3f} kg/m³")       # density
print(f"mu         = {f.mu:.3e} Pa·s")         # dynamic viscosity
print(f"nu         = {f.nu:.3e} m²/s")         # kinematic viscosity
print(f"cp         = {f.cp:.3f} J/(kg·K)")    # specific heat capacity
print(f"k          = {f.k:.3f} W/(m·K)")      # thermal conductivity
print(f"Pr         = {f.Pr:.3f}")              # Prandtl number
print(f"h          = {f.h:.3f} J/kg")         # specific enthalpy
print(f"u          = {f.u:.3f} J/kg")          # specific internal energy
print(f"s          = {f.s:.3f} J/(kg·K)")     # specific entropy
print(f"c          = {f.sound_speed:.3f} m/s") # sound speed
print(f"beta       = {f.beta:.3e} 1/K")        # isobaric expansion coefficient

rho        = 996.513 kg/m³
mu         = 8.507e-04 Pa·s
nu         = 8.537e-07 m²/s
cp         = 4180.584 J/(kg·K)
k          = 0.610 W/(m·K)
Pr         = 5.833
h          = 113323.797 J/kg
u          = 113222.118 J/kg
s          = 395.291 J/(kg·K)
c          = 1501.928 m/s
beta       = 2.763e-04 1/K
